In [1]:
# === 1. Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# === 2. Path ke file JSON dari Label Studio ===
# Ganti dengan lokasi file JSON kamu di Google Drive
input_file = "/content/drive/MyDrive/Tugas Akhir/Dataset/(2.1-BERT) Labeling BIO/Movie_review_labeledBIO.json"
output_file = "/content/drive/MyDrive/Tugas Akhir/Dataset/(2.1-BERT) Labeling BIO/Movie_review_tokenized.json"

In [7]:
# === 3. Install transformers jika belum ada ===
!pip install transformers

In [8]:
# === 4. Import library ===
import json
from transformers import BertTokenizerFast   # 🔥 ganti ke versi Fast

# Load tokenizer BERT (Fast version)
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [9]:
# === 5. Load file JSON Label Studio ===
with open(input_file, "r") as f:
    data = json.load(f)

dataset = []

In [10]:
# === 6. Proses setiap kalimat ===
for item in data:
    sentence = item["data"]["Sentence"]
    annotations = item["annotations"][0]["result"]

    # Ambil semua span aspek (start, end)
    spans = [(ann["value"]["start"], ann["value"]["end"]) for ann in annotations]

    # Tokenisasi dengan BERT Fast (bisa pakai offset_mapping)
    encoding = tokenizer(sentence, return_offsets_mapping=True, add_special_tokens=False)
    tokens = encoding.tokens()
    offsets = encoding["offset_mapping"]

    labels = []
    for (start, end), token in zip(offsets, tokens):
        label = "O"
        for span_start, span_end in spans:
            if start >= span_start and end <= span_end:
                label = "B-ASP" if start == span_start else "I-ASP"
                break
        labels.append(label)

    dataset.append({"tokens": tokens, "labels": labels})

Token indices sequence length is longer than the specified maximum sequence length for this model (795 > 512). Running this sequence through the model will result in indexing errors


In [11]:
# === 7. Simpan hasil konversi ke Drive ===
with open(output_file, "w") as f:
    json.dump(dataset, f, indent=2)

print(f"✅ Konversi selesai! Dataset tersimpan di {output_file}")

✅ Konversi selesai! Dataset tersimpan di /content/drive/MyDrive/Tugas Akhir/Dataset/(2.1-BERT) Labeling BIO/Movie_review_tokenized.json


In [12]:
# (Optional) === 8. Cek 3 data pertama untuk verifikasi ===
for sample in dataset[:3]:
    print("Tokens :", sample["tokens"])
    print("Labels :", sample["labels"])
    print("---")

Tokens : ['also', ',', 'i', 'was', 'amazed', 'by', 'the', 'great', 'moral', 'message', 'that', 'this', 'movie', 'delivered', 'to', 'us', '.']
Labels : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ASP', 'I-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
---
Tokens : ['sometimes', 'a', 'sequel', 'exceeds', 'expectations', 'and', 'sets', 'a', 'bench', '##mark', ',', 'and', 'pu', '##ss', 'in', 'boots', '2', 'has', 'done', 'exactly', 'that', '.']
Labels : ['O', 'O', 'B-ASP', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
---
Tokens : ['the', 'story', 'overall', 'is', 'great', '.']
Labels : ['O', 'B-ASP', 'O', 'O', 'O', 'O']
---


In [13]:
# === 9. Hitung jumlah aspek dan label ===
b_count = 0
i_count = 0

for sample in dataset:
    b_count += sample["labels"].count("B-ASP")
    i_count += sample["labels"].count("I-ASP")

total_aspects = b_count  # setiap aspek dimulai dengan B-ASP

print(f"📊 Jumlah aspek unik (berdasarkan B-ASP): {total_aspects}")
print(f"🔹 Jumlah token B-ASP: {b_count}")
print(f"🔹 Jumlah token I-ASP: {i_count}")
print(f"🔹 Total token berlabel aspek: {b_count + i_count}")

📊 Jumlah aspek unik (berdasarkan B-ASP): 3586
🔹 Jumlah token B-ASP: 3586
🔹 Jumlah token I-ASP: 368
🔹 Total token berlabel aspek: 3954
